# Homework 12 - Chapter 17

- Due Date: Wednesday, April 29th no later than 11:59 p.m.
- Partner Information: Learning to collaborate with others is an important skill and one that employers value.
To receive credit, this assignment requires you to partner with a classmate and submit one solution for the two of you.
- Submission Instructions: Upload your solution, entitled **YourFirstName-YourLastName-PartnerFirstName-PartnerLastName-Homework12.ipynb** to the Canvas Homework 6 Dropbox.
**Note**: only one person needs to submit a solution. If you and your partner both submit a solution, the submission that will be graded is the one from the partner whose last name comes alphabetically first.
- Deadline Reminder: Once the submission deadline passes, Canvas will no longer accept your submission and you will no longer be able to earn credit. Thus, if you are not able to fully complete the assignment, submit whatever you have before the deadline so that partial credit can be earned.

## Starting Code

In [48]:
from datascience import *
import matplotlib.pyplot as plots
%matplotlib inline
import numpy as np

Download the file pokemon.csv
into the same directory as this Jupyter notebook.

In [49]:
# Place the csv file in the same directory as your solution
pokemon = Table().read_table("pokemon.csv")
pokemon.show(3)

name,primary_type,hp,height_m,weight_kg,habitat
bulbasaur,grass,45,0.7,6.9,grassland
ivysaur,grass,60,1,13,grassland
venusaur,grass,80,2,100,grassland


## Task - 10 points

Your esteemed course assistant, Alex, loves to play Pokemon and needs your help to identify potential Pokemon buddies
that are most similar to him.  Your task is to implement a k-nearest-neighbors algorithm that Alex (or anyone) can use to
identify these buddies.

## Inputs

Your Jupyter notebook program should query the user for the following information:
* Their first name (e.g. Alex)
* The type of Pokemon that they most identify with (e.g. bug)
* Their hit points (e.g. 50)
* Their height in meters (e.g. 1.1)
* Their weight in kilograms (e.g. 60.5)
* Their preferred habitat (e.g. mountain)
* The number of Pokemon most similar to themself that they want to identify (k, an integer that can range from 1 through 100)

## Distance Calculation

It is up to you to determine how best to determine the distance between 
the user's inputs and each individual Pokemon.  However, the distance calculation
should be reasonable, incorporate the five relevant inputs and 
be explained in a comment that appears in your Jupyter notebook.  You can assume
that the user will enter valid information but do not make any assumptions
about what that information might be.

## Required Sample Output Format (for k = 2)

Alex: bug, 50, 1.1, 60.5, mountain.  Your recommended buddies are  

1. vikavolt: bug, 77, 1.5, 45, unknown (distance 3.22)
2. lunatone: rock, 90, 1, 168, cave (distance 4.81)

 Distance explanation:
 I calculate distance using the five inputs (not including name). Hit points, height, and weight are converted to standard units so they are easily comparable. Since type and habitat are categorical, I add 0 if the user's value matches the Pokemon's value and 1 if it doesn't. This helps make Pokemon with the same type/habitat closer.

In [50]:
def standard_units(numbers):
    "Convert any array of numbers to standard units."
    return (numbers - np.mean(numbers))/np.std(numbers)

def distance(point1, point2):
    return np.sqrt(np.sum((point1 - point2)**2))

def all_distances(training, new_point):
    def distance_from_point(row):
        type_distance = 0 if row.item("primary_type") == pokemon_type else 1
        hp_distance = row.item("hp (SU)") - new_point.item(0)
        height_distance = row.item("height_m (SU)") - new_point.item(1)
        weight_distance = row.item("weight_kg (SU)") - new_point.item(2)
        habitat_distance = 0 if row.item("habitat") == habitat else 1

        return np.sqrt(
            type_distance**2 +
            hp_distance**2 +
            weight_distance**2 +
            height_distance**2 +
            habitat_distance**2 
        )
    return training.apply(distance_from_point)

def table_with_distances(training, new_point):
    return training.with_column("Distance", all_distances(training, new_point))

def closest(training, new_point, k):
    with_dists = table_with_distances(training, new_point)
    sorted_by_distance = with_dists.sort("Distance")
    nearest_k = sorted_by_distance.take(np.arange(k))
    return nearest_k

def classify(training, new_point, k):
    return closest(training, new_point, k)

pokemon = Table().read_table("pokemon.csv")

name = input("Please enter your first name (e.g. Alex): ")
pokemon_type = input("Please enter the type of Pokemon do you most identify with (e.g. bug): ").lower()
hit_points = float(input("Please enter how many hit points do you have (e.g. 50): "))
height = float(input("Please enter your height in meters (e.g. 1.1): "))
weight = float(input("Please enter your weight in kilograms (e.g. 60.5): "))
habitat = input("Please enter your preferred habitat (e.g. mountain): ").lower()
k_neighbors = int(input("Please enter the number of the Pokemon you are most similar to (1-100)"))

hp_mean = np.mean(pokemon.column("hp"))
hp_std = np.std(pokemon.column("hp"))
height_mean = np.mean(pokemon.column("height_m"))
height_std = np.std(pokemon.column("height_m"))
weight_mean = np.mean(pokemon.column("weight_kg"))
weight_std = np.std(pokemon.column("weight_kg"))

pokemon = pokemon.with_columns(
    "hp (SU)", standard_units(pokemon.column("hp")),
    "height_m (SU)", standard_units(pokemon.column("height_m")),
    "weight_kg (SU)", standard_units(pokemon.column("weight_kg")),
)


special_point = make_array(
    (hit_points - hp_mean) / hp_std,
    (height - height_mean) / height_std,
    (weight - weight_mean) / weight_std
      )

predicted_friends = classify(pokemon, special_point, k_neighbors)
print(f"{name}: {pokemon_type}, {hit_points:g}, {height:g}, {weight:g}, {habitat}. Your recommended buddies are: ")
print()
for i in range(predicted_friends.num_rows):
    row = predicted_friends.row(i)
    print(f"{i+1}. {row.item('name')}: {row.item('primary_type')}, {row.item('hp'):g}, {row.item('height_m'):g}, {row.item('weight_kg'):g}, {row.item('habitat')} (distance {row.item('Distance'):.2f})")

Alex: bug, 50, 1.1, 60.5, mountain. Your recommended buddies are: 

1. larvesta: bug, 55, 1.1, 28.8, unknown (distance 1.05)
2. ledian: bug, 55, 1.4, 35.6, forest (distance 1.07)
